In [1]:
import pandas as pd
import sqlite3
import numpy as np
import warnings
warnings.simplefilter('ignore')
import logging
import time

In [2]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[
        logging.FileHandler("pipeline.log"),   
    ]
)
logger = logging.getLogger(__name__)

In [3]:
sdm = sqlite3.connect("test.sqlite")
AcV = sqlite3.connect("BikeToDrive_1_Accessoireverkoop.db")
FiV = sqlite3.connect("BikeToDrive_2_Fietsverkoop.db")
OnH = sqlite3.connect("BikeToDrive_3_Onderhoud.db")
AcI = sqlite3.connect("BikeToDrive_4_Accessoire_Inkoop.db")
FiI = sqlite3.connect("BikeToDrive_5_Fiets_Inkoop.db")

In [4]:
cur = sdm.cursor()

In [5]:
alleTabellen = [
    "Accesoireverkoop_Leverancier",
    "Accesoireverkoop_Filiaal",
    "Accesoireverkoop_Klant",
    "Accesoireverkoop_Monteur",
    "Accesoireverkoop_Accessoire",
    "Accessoire_Verkoop",
    "Fietsverkoop_Fabrikant",
    "Fietsverkoop_Filiaal",
    "Fietsverkoop_Klant",
    "Fietsverkoop_Monteur",
    "Fietsverkoop_Fiets",
    "Fiets_Verkoop",
    "Onderhoud_Fabrikant",
    "Onderhoud_Fiets",
    "Onderhoud_Filiaal",
    "Onderhoud_Monteur",
    "Onderhoud",
    "Accessoire_Inkoop_Leverancier",
    "Accessoire_Inkoop_Accessoire",
    "Accessoire_Inkoop",
    "Fiets_Inkoop_Fabrikant",
    "Fiets_Inkoop_Fiets",
    "Fiets_Inkoop"
]

In [6]:
tabellenAccessoireVerkoop = {
    "Accesoireverkoop_Leverancier": "Leverancier",
    "Accesoireverkoop_Filiaal": "Filiaal",
    "Accesoireverkoop_Klant": "Klant",
    "Accesoireverkoop_Monteur": "Monteur",
    "Accesoireverkoop_Accessoire": "Accessoire",
    "Accessoire_Verkoop" : "Accessoire_Verkoop"   
}

In [7]:
tabellenFietsVerkoop = {
    "Fietsverkoop_Fabrikant" : "Fabrikant",
    "Fietsverkoop_Filiaal" : "Filiaal",
    "Fietsverkoop_Klant" : "Klant",
    "Fietsverkoop_Monteur" : "Monteur",
    "Fietsverkoop_Fiets" : "Fiets",
    "Fiets_Verkoop" : "Fiets_Verkoop"
}

In [8]:
tabellenOnderhoud = {
    "Onderhoud_Fabrikant" : "Fabrikant",
    "Onderhoud_Fiets" : "Fiets",
    "Onderhoud_Filiaal" : "Filiaal",
    "Onderhoud_Monteur" : "Monteur",
    "Onderhoud" : "Onderhoud"
}

In [9]:
tabellenAccessoireInkoop = {
    "Accessoire_Inkoop_Leverancier" : "Leverancier",
    "Accessoire_Inkoop_Accessoire" : "Accessoire",
    "Accessoire_Inkoop" : "Accessoire_Inkoop"
}

In [10]:
tabellenFietsInkoop = {
    "Fiets_Inkoop_Fabrikant" : "Fabrikant",
    "Fiets_Inkoop_Fiets" : "Fiets",
    "Fiets_Inkoop" : "Fiets_Inkoop"
}

In [ ]:
for tabel in alleTabellen:
    start = time.time()
    logger.info(f"[{tabel}] Starting load")

    if tabel in tabellenAccessoireVerkoop:
        src = AcV
        source = tabellenAccessoireVerkoop
    elif tabel in tabellenFietsVerkoop:
        src = FiV
        source = tabellenFietsVerkoop
    elif tabel in tabellenOnderhoud:
        src = OnH
        source = tabellenOnderhoud
    elif tabel in tabellenAccessoireInkoop:
        src = AcI
        source = tabellenAccessoireInkoop
    else:
        src = FiI
        source = tabellenFietsInkoop

    try:
        df = pd.read_sql(f"SELECT * FROM {source[tabel]}", src)
        logger.info(f"[{tabel}] Extracted {len(df):,} rows from source")

        sdm.execute(f"DELETE FROM {tabel}")
        logger.info(f"[{tabel}] Deleted existing rows from target")

        df.to_sql(tabel, sdm, if_exists="append", index=False)
        logger.info(f"[{tabel}] Inserted {len(df):,} rows into target")

        sdm.commit()

        duration = round(time.time() - start, 2)
        logger.info(f"[{tabel}] Completed successfully in {duration}s")

    except Exception as e:
        sdm.rollback()
        logger.error(f"[{tabel}] Failed — {e}", exc_info=True)

In [11]:
alleTabellen = alleTabellen[::-1]
for tabel in alleTabellen:
    deleteStatement = f"DELETE FROM {tabel};"
    try:
        cur.execute(deleteStatement)
        sdm.commit()
    except sqlite3.Error as e:
        print(f"Error deleting from {tabel}: {e}")